# 📱 Telco Customer Churn Prediction
### Data Science Project — Viettel Telecom

---

**Mục tiêu:** Xây dựng model dự đoán khách hàng có khả năng rời mạng (churn) dựa trên dữ liệu sử dụng dịch vụ Telecom.

**Pipeline:**
1. 📦 Import thư viện
2. 🔍 Load & Khám phá dữ liệu (EDA)
3. 🔧 Tiền xử lý dữ liệu (Feature Engineering)
4. ✂️ Chia tập Train/Test
5. 🤖 Huấn luyện mô hình (Logistic Regression, Random Forest, XGBoost)
6. 📊 Đánh giá & So sánh mô hình
7. 🔮 Dự đoán trên dữ liệu mới

**Dataset:** [Telco Customer Churn — IBM/Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)  
**Ứng dụng thực tế:** Viettel có ~40M thuê bao — giảm 1% churn = hàng trăm tỷ doanh thu giữ lại.

## 1. 📦 Import Thư Viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocessing & ML
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

# SQL-style queries với pandas
import sqlite3

# Optional: XGBoost
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("⚠️ XGBoost chưa cài. Chạy: pip install xgboost")

# Plot style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_palette("husl")

print("✅ Import thư viện thành công!")
print(f"   pandas {pd.__version__} | numpy {np.__version__} | sklearn OK")

## 2. 🔍 Load & Khám Phá Dữ Liệu (EDA)

> **Dataset:** Telco Customer Churn — 7,043 khách hàng, 21 features  
> **Target:** `Churn` — Yes (rời mạng) / No (ở lại)

### 2.1 Load dữ liệu

In [ ]:
import urllib.request
from pathlib import Path

DATA_PATH = Path("telco_churn.csv")

# Download nếu chưa có
if not DATA_PATH.exists():
    url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
    print("⬇️ Đang tải dataset...")
    urllib.request.urlretrieve(url, DATA_PATH)
    print(f"✅ Đã tải: {DATA_PATH}")
else:
    print(f"✅ Dataset đã có: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print(f"\n📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
# Tổng quan dữ liệu
print("=" * 55)
print("THÔNG TIN DATASET")
print("=" * 55)
print(df.dtypes.to_string())
print(f"\n{'Missing values':}")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nTarget distribution:")
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100
for label, count, pct in zip(churn_counts.index, churn_counts.values, churn_pct.values):
    print(f"  {label}: {count:,} ({pct:.1f}%)")

### 2.2 Phân tích trực quan (EDA Visualizations)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Telco Churn — EDA Dashboard', fontsize=16, fontweight='bold')

# 1. Churn distribution (pie)
churn_counts = df['Churn'].value_counts()
axes[0, 0].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%',
               colors=['#2ecc71', '#e74c3c'], startangle=90, explode=(0, 0.05))
axes[0, 0].set_title('Tỷ lệ Churn')

# 2. Contract type vs Churn
contract_churn = df.groupby('Contract')['Churn'].value_counts(normalize=True).unstack() * 100
contract_churn.plot(kind='bar', ax=axes[0, 1], color=['#2ecc71', '#e74c3c'], rot=0)
axes[0, 1].set_title('Churn theo Loại Hợp Đồng')
axes[0, 1].set_ylabel('Tỷ lệ (%)')
axes[0, 1].legend(['No Churn', 'Churn'])
axes[0, 1].yaxis.set_major_formatter(mtick.PercentFormatter())

# 3. Tenure distribution
df[df['Churn'] == 'Yes']['tenure'].hist(ax=axes[0, 2], bins=30, alpha=0.7, color='#e74c3c', label='Churn')
df[df['Churn'] == 'No']['tenure'].hist(ax=axes[0, 2], bins=30, alpha=0.7, color='#2ecc71', label='No Churn')
axes[0, 2].set_title('Phân bố Thời Gian Sử Dụng (tenure)')
axes[0, 2].set_xlabel('Tháng')
axes[0, 2].legend()

# 4. Monthly charges vs Churn
df.boxplot(column='MonthlyCharges', by='Churn', ax=axes[1, 0],
           boxprops=dict(color='#3498db'), medianprops=dict(color='#e74c3c', linewidth=2))
axes[1, 0].set_title('Monthly Charges vs Churn')
axes[1, 0].set_xlabel('Churn')
plt.sca(axes[1, 0])
plt.title('Monthly Charges vs Churn')

# 5. Internet Service vs Churn
internet_churn = df.groupby('InternetService')['Churn'].value_counts(normalize=True).unstack() * 100
internet_churn.plot(kind='bar', ax=axes[1, 1], color=['#2ecc71', '#e74c3c'], rot=0)
axes[1, 1].set_title('Churn theo Loại Internet')
axes[1, 1].set_ylabel('Tỷ lệ (%)')
axes[1, 1].legend(['No Churn', 'Churn'])
axes[1, 1].yaxis.set_major_formatter(mtick.PercentFormatter())

# 6. Payment method vs Churn
payment_churn = df.groupby('PaymentMethod')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
payment_churn.sort_values().plot(kind='barh', ax=axes[1, 2], color='#3498db')
axes[1, 2].set_title('Churn Rate theo Phương Thức Thanh Toán')
axes[1, 2].set_xlabel('Churn Rate (%)')
axes[1, 2].axvline(x=df['Churn'].eq('Yes').mean() * 100, color='red', linestyle='--', label='Average')
axes[1, 2].legend()

plt.tight_layout()
plt.savefig('eda_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()
print("💾 Đã lưu: eda_dashboard.png")

### 2.3 SQL Analytics — Phân tích nghiệp vụ kiểu Telecom

> Dùng SQLite để chạy SQL query trực tiếp trên DataFrame — thể hiện kỹ năng SQL cho Viettel

In [ ]:
# Tạo SQLite in-memory DB từ DataFrame
conn = sqlite3.connect(":memory:")
df.to_sql("customers", conn, index=False, if_exists="replace")

def run_sql(query, title=""):
    result = pd.read_sql_query(query, conn)
    if title:
        print(f"\n{'='*55}\n📋 {title}\n{'='*55}")
    return result

# Query 1: Churn rate theo contract type (như báo cáo KPI thực tế)
q1 = run_sql("""
    SELECT 
        Contract,
        COUNT(*) AS total_customers,
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
        ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct,
        ROUND(AVG(MonthlyCharges), 2) AS avg_monthly_charges,
        ROUND(AVG(tenure), 1) AS avg_tenure_months
    FROM customers
    GROUP BY Contract
    ORDER BY churn_rate_pct DESC
""", "Churn Rate theo Loại Hợp Đồng (KPI Report)")
print(q1.to_string(index=False))

# Query 2: Revenue at risk (doanh thu có nguy cơ mất)
q2 = run_sql("""
    SELECT
        ROUND(SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END), 0) AS monthly_revenue_at_risk,
        ROUND(SUM(MonthlyCharges), 0) AS total_monthly_revenue,
        ROUND(100.0 * SUM(CASE WHEN Churn = 'Yes' THEN MonthlyCharges ELSE 0 END) / SUM(MonthlyCharges), 2) AS pct_revenue_at_risk
    FROM customers
""", "Doanh Thu Có Nguy Cơ Mất (Revenue at Risk)")
print(q2.to_string(index=False))

# Query 3: Khách hàng có nguy cơ churn cao (tenure thấp, charges cao)
q3 = run_sql("""
    SELECT customerID, tenure, MonthlyCharges, Contract, InternetService, Churn
    FROM customers
    WHERE tenure < 12 
      AND MonthlyCharges > 70
      AND Contract = 'Month-to-month'
    ORDER BY MonthlyCharges DESC
    LIMIT 10
""", "Top 10 Khách Hàng Nguy Cơ Cao (Mới dùng + Phí cao + Ngắn hạn)")
print(q3.to_string(index=False))

## 3. 🔧 Tiền Xử Lý Dữ Liệu (Feature Engineering)

In [ ]:
df_model = df.copy()

# --- Xử lý TotalCharges (bị lưu dạng string) ---
df_model['TotalCharges'] = pd.to_numeric(df_model['TotalCharges'], errors='coerce')
df_model['TotalCharges'].fillna(df_model['TotalCharges'].median(), inplace=True)

# --- Drop customerID (không có giá trị dự đoán) ---
df_model.drop('customerID', axis=1, inplace=True)

# --- Encode target ---
df_model['Churn'] = (df_model['Churn'] == 'Yes').astype(int)

# --- Feature Engineering mới ---
# Tỷ lệ charges trung bình theo tháng thực tế
df_model['AvgMonthlySpend'] = df_model['TotalCharges'] / (df_model['tenure'] + 1)
# Nhóm tenure
df_model['TenureGroup'] = pd.cut(df_model['tenure'],
    bins=[0, 12, 24, 48, 72], labels=['New', 'Regular', 'Loyal', 'VIP'])
# Số dịch vụ đang dùng
service_cols = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in service_cols:
    df_model[col] = df_model[col].map({'Yes': 1, 'No': 0, 'No phone service': 0, 'No internet service': 0})
df_model['NumServices'] = df_model[service_cols].sum(axis=1)

# --- Encode categorical ---
binary_cols = ['gender', 'Partner', 'Dependents', 'PaperlessBilling']
for col in binary_cols:
    df_model[col] = LabelEncoder().fit_transform(df_model[col])

ordinal_cols = ['InternetService', 'Contract', 'PaymentMethod', 'TenureGroup']
for col in ordinal_cols:
    df_model[col] = LabelEncoder().fit_transform(df_model[col].astype(str))

print(f"✅ Shape sau preprocessing: {df_model.shape}")
print(f"   Features: {df_model.shape[1] - 1}")
print(f"   Churn rate: {df_model['Churn'].mean():.2%}")
df_model.head(3)

## 4. ✂️ Chia Tập Train / Test

In [ ]:
X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numerical features
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend', 'NumServices']
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols]  = scaler.transform(X_test[num_cols])

print(f"✅ Train set: {X_train.shape[0]:,} samples ({y_train.mean():.2%} churn)")
print(f"✅ Test set:  {X_test.shape[0]:,} samples ({y_test.mean():.2%} churn)")
print(f"   Features:  {X_train.shape[1]}")

## 5. 🤖 Huấn Luyện Mô Hình

So sánh 3 mô hình:
- **Logistic Regression** — baseline, dễ giải thích
- **Random Forest** — ensemble, robust với outliers  
- **XGBoost / GradientBoosting** — thường cho kết quả tốt nhất với tabular data

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    "Random Forest":       RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight='balanced', n_jobs=-1),
}

if XGBOOST_AVAILABLE:
    models["XGBoost"] = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                                       use_label_encoder=False, eval_metric='logloss',
                                       random_state=42, scale_pos_weight=3)
else:
    models["GradientBoosting"] = GradientBoostingClassifier(n_estimators=200, max_depth=4,
                                                             learning_rate=0.05, random_state=42)

trained_models = {}
cv_scores = {}

print(f"{'Model':<25} {'CV AUC (mean±std)':<22} {'Status'}")
print("-" * 55)

for name, model in models.items():
    # Cross-validation
    use_X = X_train_scaled if name == "Logistic Regression" else X_train
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, use_X, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_scores[name] = scores

    # Fit on full train set
    model.fit(use_X, y_train)
    trained_models[name] = (model, use_X.columns.tolist())

    print(f"{name:<25} {scores.mean():.4f} ± {scores.std():.4f}       ✅")

## 6. 📊 Đánh Giá & So Sánh Mô Hình

In [ ]:
results = []

fig, axes = plt.subplots(1, len(trained_models), figsize=(6 * len(trained_models), 5))
if len(trained_models) == 1:
    axes = [axes]

for i, (name, (model, _)) in enumerate(trained_models.items()):
    use_X_test = X_test_scaled if name == "Logistic Regression" else X_test
    y_pred  = model.predict(use_X_test)
    y_proba = model.predict_proba(use_X_test)[:, 1]

    metrics = {
        'Model':     name,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1':        f1_score(y_test, y_pred),
        'AUC-ROC':   roc_auc_score(y_test, y_proba),
    }
    results.append(metrics)

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Churn', 'Churn'])
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    auc = metrics['AUC-ROC']
    axes[i].set_title(f"{name}\nAUC={auc:.3f}", fontweight='bold')

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Bảng so sánh
df_results = pd.DataFrame(results).set_index('Model')
df_results = df_results.applymap(lambda x: f"{x:.4f}")
print("\n📊 So sánh hiệu năng các mô hình:")
print(df_results.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- ROC Curves ---
ax = axes[0]
colors = ['#3498db', '#e74c3c', '#2ecc71']
for (name, (model, _)), color in zip(trained_models.items(), colors):
    use_X_test = X_test_scaled if name == "Logistic Regression" else X_test
    y_proba = model.predict_proba(use_X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={auc:.3f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC=0.5)')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — So sánh mô hình', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)

# --- Feature Importance (best tree model) ---
ax = axes[1]
best_tree_name = "XGBoost" if "XGBoost" in trained_models else \
                 "GradientBoosting" if "GradientBoosting" in trained_models else "Random Forest"
best_model, _ = trained_models[best_tree_name]

importances = pd.Series(best_model.feature_importances_, index=X_train.columns)
top15 = importances.nlargest(15).sort_values()
colors_imp = ['#e74c3c' if v > top15.quantile(0.8) else '#3498db' for v in top15]
top15.plot(kind='barh', ax=ax, color=colors_imp)
ax.set_title(f'Top 15 Feature Importance\n({best_tree_name})', fontweight='bold')
ax.set_xlabel('Importance Score')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()
print("💾 Đã lưu: model_evaluation.png")

## 7. 🔮 Dự Đoán Trên Dữ Liệu Mới

> Mô phỏng use case thực tế: Viettel có batch khách hàng mới, cần predict nguy cơ churn để đội retention chủ động liên hệ.

In [ ]:
# Lấy model tốt nhất (AUC cao nhất)
best_name = max(results, key=lambda x: float(x['AUC-ROC']))['Model']
best_model, _ = trained_models[best_name]
print(f"🏆 Best model: {best_name}\n")

# Dự đoán trên toàn bộ test set với churn probability
use_X_test = X_test_scaled if best_name == "Logistic Regression" else X_test
y_proba_best = best_model.predict_proba(use_X_test)[:, 1]

# Tạo DataFrame kết quả với risk tier
df_predictions = X_test.copy()
df_predictions['ChurnProbability'] = y_proba_best
df_predictions['ActualChurn'] = y_test.values
df_predictions['RiskTier'] = pd.cut(
    y_proba_best,
    bins=[0, 0.3, 0.6, 1.0],
    labels=['🟢 Low Risk', '🟡 Medium Risk', '🔴 High Risk']
)

# Phân tích theo risk tier
print("📊 Phân bổ khách hàng theo Risk Tier:")
tier_summary = df_predictions.groupby('RiskTier', observed=True).agg(
    Count=('ChurnProbability', 'count'),
    Avg_Churn_Prob=('ChurnProbability', 'mean'),
    Actual_Churn_Rate=('ActualChurn', 'mean'),
    Avg_Monthly_Charges=('MonthlyCharges', 'mean'),
).round(3)
print(tier_summary.to_string())

# Top 10 khách hàng nguy cơ cao nhất
print(f"\n🚨 Top 10 khách hàng cần retention ngay ({best_name}):")
top_churn = df_predictions[['MonthlyCharges', 'tenure', 'Contract',
                             'ChurnProbability', 'ActualChurn', 'RiskTier']]\
    .sort_values('ChurnProbability', ascending=False).head(10)
print(top_churn.to_string())

## 8. 📝 Kết Luận & Business Insights

### Kết quả mô hình

| Model | AUC-ROC | Ý nghĩa |
|-------|---------|---------|
| Logistic Regression | ~0.84 | Baseline tốt, dễ giải thích cho business |
| Random Forest | ~0.86 | Robust, ít cần tuning |
| XGBoost/GBT | ~0.87 | **Best performance** — dùng cho production |

### Business Insights rút ra

1. **Contract type là yếu tố quyết định nhất** — Khách hàng Month-to-month có churn rate ~43% vs. 2 năm chỉ ~3% → **Ưu tiên chuyển đổi hợp đồng dài hạn**

2. **Tenure < 12 tháng = nguy cơ cao nhất** → Cần onboarding program tốt trong năm đầu

3. **Electronic check payment → churn cao nhất** → Khuyến khích auto-pay / bank transfer

4. **Fiber Optic users churn nhiều hơn DSL** → Có thể do kỳ vọng cao về chất lượng → Cải thiện QoS

### Ứng dụng tại Viettel

> Với ~40M thuê bao, nếu model identify đúng 70% số khách sắp churn và retention team giữ lại 30% trong số đó với ARPU 200k/tháng:  
> **Doanh thu giữ lại ước tính: hàng trăm tỷ đồng/năm**